# Coleta de Dados — CAGED Telemarketing

Este notebook coleta os dados de emprego formal do setor de telemarketing 
a partir do CAGED via Base dos Dados (BigQuery).

**Fonte:** Ministério do Trabalho e Emprego — CAGED  
**Acesso:** Base dos Dados (basedosdados.org)  
**Ocupação:** Operador de Telemarketing (CBO 4223)

In [4]:
# Importações
import basedosdados as bd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


In [5]:
import os

# Define o projeto que vamos usar no BigQuery
os.environ['GOOGLE_CLOUD_PROJECT'] = 'telemarketing-decline'

print("Projeto configurado!")

Projeto configurado!


In [18]:
query = """
SELECT
    ano,
    mes,
    SUM(saldo_movimentacao) as saldo_empregos
FROM `basedosdados.br_me_caged.microdados_movimentacao`
WHERE
    cbo_2002 LIKE '4223%'
    AND ano BETWEEN 2020 AND 2025
GROUP BY
    ano, mes
ORDER BY
    ano, mes
"""

print("Query definida!")
print(query)

Query definida!

SELECT
    ano,
    mes,
    SUM(saldo_movimentacao) as saldo_empregos
FROM `basedosdados.br_me_caged.microdados_movimentacao`
WHERE
    cbo_2002 LIKE '4223%'
    AND ano BETWEEN 2020 AND 2025
GROUP BY
    ano, mes
ORDER BY
    ano, mes



In [19]:
df = bd.read_sql(
    query=query,
    billing_project_id="telemarketing-decline"
)

print("Dados carregados!")
print(f"Total de linhas: {len(df)}")
print(df.head())

Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|
Dados carregados!
Total de linhas: 67
    ano  mes  saldo_empregos
0  2020    1           -2144
1  2020    2            -980
2  2020    3           -8530
3  2020    4          -14504
4  2020    5            4882


In [20]:
print("Shape:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)
print("\nPrimeiras linhas:")
df.head(10)

Shape: (67, 3)

Tipos de dados:
ano               Int64
mes               Int64
saldo_empregos    Int64
dtype: object

Primeiras linhas:


,ano,mes,saldo_empregos
0,2020,1,-2144
1,2020,2,-980
2,2020,3,-8530
3,2020,4,-14504
4,2020,5,4882
5,2020,6,10299
6,2020,7,4189
7,2020,8,3775
8,2020,9,5530
9,2020,10,10574


In [21]:
print("Anos disponíveis:", sorted(df['ano'].unique()))
print("Meses disponíveis:", sorted(df['mes'].unique()))
print("\nTotal de registros:", len(df))
print("\nEstatísticas do saldo:")
print(df['saldo_empregos'].describe())

Anos disponíveis: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Meses disponíveis: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]

Total de registros: 67

Estatísticas do saldo:
count           67.0
mean      854.089552
std      4532.353609
min         -14504.0
25%          -1367.5
50%            255.0
75%           3536.5
max          14181.0
Name: saldo_empregos, dtype: Float64


In [22]:
query_antiga = """
SELECT
    ano,
    mes,
    SUM(saldo_movimentacao) as saldo_empregos
FROM `basedosdados.br_me_caged.microdados`
WHERE
    cbo_2002 LIKE '4223%'
    AND ano BETWEEN 2010 AND 2019
GROUP BY
    ano, mes
ORDER BY
    ano, mes
"""

df_antigo = bd.read_sql(
    query=query_antiga,
    billing_project_id="telemarketing-decline"
)

print("Dados antigos carregados!")
print(f"Total de linhas: {len(df_antigo)}")
print("Anos disponíveis:", sorted(df_antigo['ano'].unique()))

GenericGBQException: Reason: 404 POST https://bigquery.googleapis.com/bigquery/v2/projects/telemarketing-decline/queries?prettyPrint=false: Not found: Table basedosdados:br_me_caged.microdados was not found in location US

In [23]:
query_catalogo = """
SELECT table_name
FROM `basedosdados.br_me_caged.INFORMATION_SCHEMA.TABLES`
"""

df_catalogo = bd.read_sql(
    query=query_catalogo,
    billing_project_id="telemarketing-decline"
)

print(df_catalogo)

Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|
                           table_name
0    microdados_movimentacao_excluida
1                  microdados_antigos
2  microdados_movimentacao_fora_prazo
3                          dicionario
4          microdados_antigos_ajustes
5             microdados_movimentacao


In [24]:
query_colunas = """
SELECT column_name
FROM `basedosdados.br_me_caged.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'microdados_antigos'
"""

df_colunas = bd.read_sql(
    query=query_colunas,
    billing_project_id="telemarketing-decline"
)

print(df_colunas)

Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|
                        column_name
0                               ano
1                               mes
2                          sigla_uf
3                      id_municipio
4                    id_municipio_6
5              admitidos_desligados
6              tipo_estabelecimento
7     tipo_movimentacao_desagregado
8      faixa_emprego_inicio_janeiro
9                     tempo_emprego
10     quantidade_horas_contratadas
11                   salario_mensal
12               saldo_movimentacao
13               indicador_aprendiz
14  indicador_trabalho_intermitente
15       indicador_trabalho_parcial
16   indicador_portador_deficiencia
17                 tipo_deficiencia
18                         cbo_2002
19                           cnae_1
20                    

In [25]:
query_antiga = """
SELECT
    ano,
    mes,
    SUM(saldo_movimentacao) as saldo_empregos
FROM `basedosdados.br_me_caged.microdados_antigos`
WHERE
    cbo_2002 LIKE '4223%'
    AND ano BETWEEN 2010 AND 2019
GROUP BY
    ano, mes
ORDER BY
    ano, mes
"""

df_antigo = bd.read_sql(
    query=query_antiga,
    billing_project_id="telemarketing-decline"
)

print("Dados históricos carregados!")
print(f"Total de linhas: {len(df_antigo)}")
print("Anos disponíveis:", sorted(df_antigo['ano'].unique()))
df_antigo.head()

Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|
Dados históricos carregados!
Total de linhas: 120
Anos disponíveis: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]


,ano,mes,saldo_empregos
0,2010,1,1998
1,2010,2,3405
2,2010,3,8439
3,2010,4,4311
4,2010,5,6213


In [26]:
df_completo = pd.concat([df_antigo, df], ignore_index=True)

# Verificar resultado
print(f"Registros históricos (2010-2019): {len(df_antigo)}")
print(f"Registros novos (2020-2024): {len(df)}")
print(f"Total combinado: {len(df_completo)}")
print(f"\nAnos disponíveis: {sorted(df_completo['ano'].unique())}")

Registros históricos (2010-2019): 120
Registros novos (2020-2024): 67
Total combinado: 187

Anos disponíveis: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [27]:
# Salvar os dados brutos na pasta raw
df_completo.to_csv('../data/raw/caged_telemarketing_raw.csv', index=False)

print("Dados salvos em data/raw/caged_telemarketing_raw.csv")
print(f"Shape: {df_completo.shape}")
df_completo.head()

Dados salvos em data/raw/caged_telemarketing_raw.csv
Shape: (187, 3)


,ano,mes,saldo_empregos
0,2010,1,1998
1,2010,2,3405
2,2010,3,8439
3,2010,4,4311
4,2010,5,6213
